# GRADIENT BOOSTING

__INDEX__
1. [Understanding Gradient Tree Boosting](#understanding)
2. [Setup and Data Preparation](#setup)
3. [Cross-Validation Framework](#crossvalframe)
4. [Experimental Runs](#expruns)
5. [Final Configuration](#finalconfig)

## 1. Understanding Gradient Tree Boosting
<a id="understanding"></a>

- Gradient-boosted trees consists in an algorithm that ensembles the predictions of weker prediction models, particularly decision trees. Each of these predictors will correct their predecessor. Unlike many algorithms, this one doesn't just deal with improving weights at each iteration, instead tries to learn frrom the previous predictors residual errors and fit the following predictor to those.
- This algorithm can be seen as an optimization one bassed on a certain loss function. 

### 1.2. Gradient Tree Boosting (GTB) in Scikit learn

- Scikit-learn (0.21) provides us with two options to implement this algorithm (they are availabl both for classification and regression, but we will only focus on the regression ones), both of them inspired by LightGBM:
    - __'HistGradientBoostingRegressor'__ and
    - __'GradientBoostingRegressor'__
- An interesting detail about this implementations is that they support missing values and for the __'HistGradientBoostingRegressor'__ one categorical data is supported aswell. 
- For small datasets, the use of bins might contribute to very close split points, ence why, in this case, __'GradientBoostingRegressor'__ is preferible. However, our dataset granularity is moderatly high and, as indicated by the official documentation, for a sample with tens of thousands of samples, __'HistGradientBoostingRegressor'__ is preferible. 
- In addition, __'HistGradientBoostingRegressor'__ has the advantage of increased time efficiency and complexity (that will be later on explained in the notebook) compared to __'GradientBoostingRegressor'__. However, the former does not natively support some features of the latter. 

Note: all of these informations are available at the scikit learn documentation.

### 1.3. Understanding __'HistGradientBoostingRegressor'__

- We decided to use this algorithm because of the infra citated reasons as well as because of the reasons citated in ths section. 

## 2. Setup and Data Preparation  <a id="setup"></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
import os
from sklearn.feature_selection import SelectFromModel
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel



In [3]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# at this point, this are all the features in use 
categorical_features = ['Brand', 'model', 'transmission', 'fuelType']              
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

N_SPLITS = 8 # the number of folds for K-Fold cross-validation
RANDOM_STATE = 42 # seed to control randomness
# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 10
RANDOM_STATE = 42


Exception: File `'05_0_preproc_helpers.ipynb'` not found.

## 3. Cross-Validation Framework
<a id="crossvalframe"></a>

## 4. Experimental Runs  <a id="expruns"></a>

### 4.1. All Features Baseline 

- In this section, we performed a random search to evaluate if the parameters search space was good and to use this values as a baseline to access improvement (or not) after. 

#### 4.1.1. Without Feature Selection

In [ ]:
# ---------------------------------------------------------
# 2. CONFIGURAÇÃO DO CROSS-VALIDATION
# ---------------------------------------------------------
N_SPLITS = 8 
RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# ---------------------------------------------------------
# 3. HIPERPARÂMETROS
# ---------------------------------------------------------
param_distributions = {
    "max_iter": [600, 800, 1000],
    "learning_rate": [0.05, 0.1, 0.15],
    "max_leaf_nodes": [31, 63, 127],
    "max_depth": [None, 10, 20, 11, 15],
    "l2_regularization": [0, 1, 2, 5],
    "min_samples_leaf": [20, 40]
}

# ---------------------------------------------------------
# 4. CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
N_RANDOM_CONFIGS = 10
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = [] 
best_rmse = np.inf 
best_config = None 

# --- NOVO: Configuração da Pasta de Logs ---
log_dir = "gb_logs"
os.makedirs(log_dir, exist_ok=True)  # Cria a pasta se não existir
log_filename = "2gb_complete_nof_log.txt"
log_path = os.path.join(log_dir, log_filename)
# -------------------------------------------

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    log("# =============================")
    log(f"# START OF HGB SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")
    
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        # Listas para métricas (Validation)
        fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
        # Listas para métricas (Train)
        fold_rmses_tr, fold_maes_tr, fold_r2s_tr, fold_bias_tr = [], [], [], []
        
        fold_med_ae = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # --- A) Split Data ---
            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

            log(f"\n[C{config_id}|F{fold}] Processing fold...")

            # --- B) Pipeline de Limpeza ---
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # Tax Custom Rules (Stateless)
            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            # Resolvers
            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            # --- C) Encoding ---
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # --- D) Matriz Final ---
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            # =========================================================
            # LOG DAS FEATURES (Imprimir 1 vez por config)
            # =========================================================
            if fold == 1:
                feature_names = list(X_train_final.columns)
                log(f"  > Features Used ({len(feature_names)}): {feature_names}")
            # =========================================================
            
            # --- E) Treino ---
            hgb_model = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params)
            hgb_model.fit(X_train_final, y_train)

            # --- F) Previsões ---
            y_pred_train = hgb_model.predict(X_train_final)
            y_pred_val   = hgb_model.predict(X_val_final)

            # --- G) Métricas Detalhadas ---
            
            # 1. TRAIN Metrics
            mae_tr = mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_tr = r2_score(y_train, y_pred_train)
            bias_tr = np.mean(y_train - y_pred_train)

            # 2. VALIDATION Metrics
            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            r2_val = r2_score(y_val, y_pred_val)
            bias_val = np.mean(y_val - y_pred_val)
            med_ae_val = median_absolute_error(y_val, y_pred_val)

            # LOG ATUALIZADO (ORDEM: R2, RMSE, MAE, BIAS)
            log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
            log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

            # Store per fold
            fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
            fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
            fold_med_ae.append(med_ae_val)

        # Average metrics
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)
        
        mean_mae_tr   = np.mean(fold_maes_tr)
        mean_r2_tr    = np.mean(fold_r2s_tr)
        mean_bias_tr  = np.mean(fold_bias_tr)
        
        log("")
        log(f"Config {config_id} SUMMARY:")
        log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
        log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

        search_results.append({
            "config_id": config_id,
            **params,
            # Validation
            "val_rmse": mean_rmse_val,
            "val_mae": mean_mae_val,
            "val_r2": mean_r2_val,
            "val_bias": mean_bias_val,
            # Train
            "train_mae": mean_mae_tr,
            "train_r2": mean_r2_tr,
            "train_bias": mean_bias_tr,
            # Robustness
            "val_med_ae": np.mean(fold_med_ae)
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF SEARCH")

# -----------------------------
# Results
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="val_rmse", ascending=True)

print("\nTop 5 Configurations by RMSE:")
cols = ["config_id", "learning_rate", "max_iter", "val_rmse", "val_mae", "val_r2", "train_r2", "val_bias"]
display(results_df_sorted[cols].head(5))

print("\nBest Config found:")
print(best_config)

# =============================
# START OF HGB SEARCH (DETAILED METRICS: R2, RMSE, MAE, BIAS)
# LOG FILE SAVED TO: gb_logs/2gb_complete_nof_log.txt
# =============================

######## CONFIG 1/10 ########
Parameters: {'min_samples_leaf': 20, 'max_leaf_nodes': 63, 'max_iter': 1000, 'max_depth': 20, 'learning_rate': 0.05, 'l2_regularization': 5}

[C1|F1] Processing fold...
  > Features Used (16): ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER', 'fuelType_PETROL']
  > [TRAIN] R2: 0.9612 | RMSE: 1924 | MAE: 1181 | Bias: 6.3
  > [VAL]   R2: 0.9563 | RMSE: 1990 | MAE: 1309 | Bias: -36.9

[C1|F2] Processing fold...
  > [TRAIN] R2: 0.9679 | RMSE: 1746 | MAE: 1139 | Bias: 1.1
  > [VAL]   R2: 0.9498 | RMSE: 2169 | MAE: 1323 | Bias: -11.4

[C1|F3] Processing fold...
  > [TRAIN] R2: 0.9629 | RMSE: 1877 | MAE:

,config_id,learning_rate,max_iter,val_rmse,val_mae,val_r2,train_r2,val_bias
7,8,0.10,600,2134.871525,1283.759934,0.951767,0.971301,4.099316
3,4,0.15,1000,2140.256753,1288.369477,0.951520,0.971999,3.652892
0,1,0.05,1000,2153.685454,1302.447808,0.950903,0.966652,3.115471
1,2,0.15,600,2177.221894,1338.780521,0.949841,0.962922,-0.559954
5,6,0.05,1000,2194.741353,1320.073472,0.949063,0.962856,1.958502



Best Config found:
{'min_samples_leaf': 20, 'max_leaf_nodes': 127, 'max_iter': 600, 'max_depth': 20, 'learning_rate': 0.1, 'l2_regularization': 0}


#### 4.1.2. With Feature Selection (experimenting different max features value)

Here, we chose our 3 best configurations and run with different max feature values to see the best strategy. This was not a very exaustive search, because again these values have the porpuse of serving as the baseline. 

### 4.2. Feature Engineering and Feature Selection configurations

#### 4.2.1. Without Feature Selection

- We did a very small random search without feature selection, with the a priori expectation of a not very good result since many features with induce redundancy.

In [ ]:
# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------
logging.basicConfig(
    filename="random_search_results_testing.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w"
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)
print(f"Geradas {len(CONFIGS)} configurações aleatórias. A iniciar testes...")

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Estas listas assumem que já as tens definidas no notebook:
# - numeric_features
# - categorical_features
# - fill_unknown
# - column_string_transformer
# - fit_* e transform_* (os teus passos)
#
# NOTA: Mantemos previousOwners e mileage durante cleaning, e só drop depois.

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING (dependente de treino -> fit no treino, apply no val)
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        # previousOwners: NÃO DROPAR antes disto
        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING (NOVAS FEATURES)
        # -----------------------------------------------------

        # 1) owners_flagged e DROP previousOwners (depois do imputer)
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        # 2) age e DROP year
        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        # 3) mileage features e DROP mileage (nota: garante drop)
        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        # 4) engine bins (mantém engineSize)
        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # Tratar engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        # Atualiza low_card para incluir engine_bin
        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        # Target Encoding (alta cardinalidade)
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        # One-Hot (baixa cardinalidade + engine_bin)
        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        # Numéricas atuais = tudo o que não está nas cats usadas
        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # E) TREINO
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_final, y_train_log)

        pred_tr_log  = model.predict(X_train_final)
        pred_val_log = model.predict(X_val_final)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f}")
    logging.info(f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | Params: {params}")

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n--- A processar {i+1}/{len(CONFIGS)} ---")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n=== TOP 10 MELHORES CONFIGURAÇÕES ===")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_final_results.csv", index=False)
print("Guardado: random_search_final_results.csv")
print("Guardado log: random_search_results.log")


Geradas 10 configurações aleatórias. A iniciar testes...

--- A processar 1/10 ---
   >> [R0_iter600_lr0.084_l20.50_leaf31] Train MAE: 1214.35 | Val MAE: 1322.45
   >> [R1_iter800_lr0.146_l21.03_leaf127] Train MAE: 1116.98 | Val MAE: 1289.89
   >> [R2_iter1500_lr0.087_l20.92_leaf127] Train MAE: 996.58 | Val MAE: 1256.58
   >> [R3_iter1500_lr0.146_l22.50_leaf127] Train MAE: 1033.52 | Val MAE: 1272.90
   >> [R4_iter1500_lr0.128_l20.82_leaf63] Train MAE: 1075.44 | Val MAE: 1272.16

--- A processar 6/10 ---
   >> [R5_iter800_lr0.080_l21.66_leaf63] Train MAE: 1141.81 | Val MAE: 1289.35
   >> [R6_iter1000_lr0.106_l21.34_leaf127] Train MAE: 1039.14 | Val MAE: 1265.58
   >> [R7_iter1500_lr0.139_l22.29_leaf31] Train MAE: 1198.26 | Val MAE: 1315.55
   >> [R8_iter600_lr0.084_l21.66_leaf127] Train MAE: 1072.79 | Val MAE: 1274.83
   >> [R9_iter1500_lr0.080_l21.13_leaf63] Train MAE: 1096.70 | Val MAE: 1272.69

=== TOP 10 MELHORES CONFIGURAÇÕES ===


,config,val_mae_mean,train_mae_mean,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
2,R2_iter1500_lr0.087_l20.92_leaf127,1256.578731,996.576759,0.087368,1500,0.921053,127,20.0,20
6,R6_iter1000_lr0.106_l21.34_leaf127,1265.576273,1039.136049,0.105789,1000,1.342105,127,NaN,30
4,R4_iter1500_lr0.128_l20.82_leaf63,1272.155654,1075.442789,0.127895,1500,0.815789,63,NaN,30
9,R9_iter1500_lr0.080_l21.13_leaf63,1272.686265,1096.698205,0.080000,1500,1.131579,63,20.0,20
3,R3_iter1500_lr0.146_l22.50_leaf127,1272.903714,1033.520589,0.146316,1500,2.500000,127,25.0,30
8,R8_iter600_lr0.084_l21.66_leaf127,1274.833602,1072.788230,0.083684,600,1.657895,127,10.0,20
5,R5_iter800_lr0.080_l21.66_leaf63,1289.349284,1141.807531,0.080000,800,1.657895,63,15.0,40
1,R1_iter800_lr0.146_l21.03_leaf127,1289.892316,1116.975504,0.146316,800,1.026316,127,10.0,40
7,R7_iter1500_lr0.139_l22.29_leaf31,1315.552940,1198.258439,0.138947,1500,2.289474,31,15.0,40
0,R0_iter600_lr0.084_l20.50_leaf31,1322.451619,1214.351604,0.083684,600,0.500000,31,10.0,30


Guardado: random_search_final_results.csv
Guardado log: random_search_results.log


#### 4.2.2. With Feature Selection

- In this experiments, we perfomed a random search with different percentages of most useful features to access the best features selection strategy. 

##### __80%__ 

In [ ]:
# este 
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 15
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.80

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results.csv", index=False)
print("stored at: random_search_fe_fs_results.csv")
print(" log in: random_search_fe_fs_results.log")



-- doing 1/15
   >> [R0_iter800_lr0.150_l21.34_leaf63] Train MAE: 1150.19 | Val MAE: 1301.44 | Sel: 20 feats
   >> [R1_iter1500_lr0.117_l22.08_leaf127] Train MAE: 1049.87 | Val MAE: 1267.03 | Sel: 20 feats
   >> [R2_iter800_lr0.095_l21.24_leaf31] Train MAE: 1154.68 | Val MAE: 1295.28 | Sel: 20 feats
   >> [R3_iter600_lr0.098_l22.18_leaf127] Train MAE: 1040.17 | Val MAE: 1263.01 | Sel: 20 feats
   >> [R4_iter1500_lr0.124_l21.66_leaf127] Train MAE: 1044.36 | Val MAE: 1268.02 | Sel: 20 feats

-- doing 6/15
   >> [R5_iter600_lr0.117_l22.39_leaf31] Train MAE: 1168.81 | Val MAE: 1298.09 | Sel: 20 feats
   >> [R6_iter1000_lr0.080_l21.97_leaf63] Train MAE: 1115.98 | Val MAE: 1277.13 | Sel: 20 feats
   >> [R7_iter600_lr0.080_l20.92_leaf31] Train MAE: 1209.38 | Val MAE: 1316.44 | Sel: 20 feats
   >> [R8_iter800_lr0.098_l21.13_leaf63] Train MAE: 1088.03 | Val MAE: 1269.95 | Sel: 20 feats
   >> [R9_iter800_lr0.113_l21.03_leaf63] Train MAE: 1100.94 | Val MAE: 1278.36 | Sel: 20 feats

-- doing 11/1

,config,val_mae_mean,train_mae_mean,avg_selected_features,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
10,R10_iter1000_lr0.091_l21.76_leaf127,1252.630196,1009.152409,20.0,0.091053,1000,1.763158,127,NaN,20
3,R3_iter600_lr0.098_l22.18_leaf127,1263.013510,1040.173371,20.0,0.098421,600,2.184211,127,20.0,30
1,R1_iter1500_lr0.117_l22.08_leaf127,1267.032613,1049.868542,20.0,0.116842,1500,2.078947,127,15.0,30
4,R4_iter1500_lr0.124_l21.66_leaf127,1268.020348,1044.361099,20.0,0.124211,1500,1.657895,127,15.0,30
8,R8_iter800_lr0.098_l21.13_leaf63,1269.949345,1088.025548,20.0,0.098421,800,1.131579,63,25.0,30
14,R14_iter600_lr0.139_l20.71_leaf127,1277.007332,1079.741646,20.0,0.138947,600,0.710526,127,15.0,40
6,R6_iter1000_lr0.080_l21.97_leaf63,1277.132114,1115.978164,20.0,0.080000,1000,1.973684,63,15.0,40
9,R9_iter800_lr0.113_l21.03_leaf63,1278.363524,1100.935909,20.0,0.113158,800,1.026316,63,25.0,30
13,R13_iter1500_lr0.146_l20.82_leaf63,1287.216976,1113.303473,20.0,0.146316,1500,0.815789,63,20.0,40
2,R2_iter800_lr0.095_l21.24_leaf31,1295.281678,1154.675361,20.0,0.094737,800,1.236842,31,25.0,20


stored at: random_search_fe_fs_results.csv
 log in: random_search_fe_fs_results.log


##### __85%__ 

In [ ]:
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 15
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.85

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results_85.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results_85.csv", index=False)
print("stored at: random_search_fe_fs_results_85.csv")
print(" log in: random_search_fe_fs_results_85.log")



-- doing 1/15
   >> [R0_iter600_lr0.109_l22.39_leaf127] Train MAE: 1033.95 | Val MAE: 1261.15 | Sel: 22 feats
   >> [R1_iter1500_lr0.109_l21.13_leaf127] Train MAE: 1119.99 | Val MAE: 1291.19 | Sel: 22 feats
   >> [R2_iter800_lr0.102_l22.18_leaf63] Train MAE: 1110.00 | Val MAE: 1278.37 | Sel: 22 feats
   >> [R3_iter800_lr0.098_l21.55_leaf63] Train MAE: 1124.44 | Val MAE: 1285.85 | Sel: 22 feats
   >> [R4_iter1500_lr0.121_l21.87_leaf31] Train MAE: 1139.62 | Val MAE: 1288.76 | Sel: 22 feats

-- doing 6/15
   >> [R5_iter1500_lr0.080_l21.66_leaf127] Train MAE: 1079.51 | Val MAE: 1271.39 | Sel: 22 feats
   >> [R6_iter1000_lr0.146_l20.92_leaf31] Train MAE: 1177.45 | Val MAE: 1308.59 | Sel: 22 feats
   >> [R7_iter600_lr0.109_l22.39_leaf63] Train MAE: 1112.38 | Val MAE: 1281.58 | Sel: 22 feats
   >> [R8_iter800_lr0.095_l22.50_leaf127] Train MAE: 1042.85 | Val MAE: 1264.37 | Sel: 22 feats
   >> [R9_iter800_lr0.091_l21.24_leaf127] Train MAE: 1039.44 | Val MAE: 1262.52 | Sel: 22 feats

-- doing 1

,config,val_mae_mean,train_mae_mean,avg_selected_features,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
0,R0_iter600_lr0.109_l22.39_leaf127,1261.151179,1033.949108,22.0,0.109474,600,2.394737,127,20.0,30
9,R9_iter800_lr0.091_l21.24_leaf127,1262.516028,1039.439003,22.0,0.091053,800,1.236842,127,15.0,30
12,R12_iter1500_lr0.091_l22.08_leaf127,1263.808871,1045.245355,22.0,0.091053,1500,2.078947,127,NaN,20
8,R8_iter800_lr0.095_l22.50_leaf127,1264.371280,1042.846505,22.0,0.094737,800,2.500000,127,20.0,20
11,R11_iter1000_lr0.146_l21.66_leaf127,1269.086650,1027.890915,22.0,0.146316,1000,1.657895,127,20.0,20
5,R5_iter1500_lr0.080_l21.66_leaf127,1271.392670,1079.508578,22.0,0.080000,1500,1.657895,127,NaN,40
2,R2_iter800_lr0.102_l22.18_leaf63,1278.372020,1109.995980,22.0,0.102105,800,2.184211,63,NaN,30
13,R13_iter800_lr0.135_l22.18_leaf63,1279.305564,1090.580480,22.0,0.135263,800,2.184211,63,25.0,20
7,R7_iter600_lr0.109_l22.39_leaf63,1281.583236,1112.379328,22.0,0.109474,600,2.394737,63,NaN,40
3,R3_iter800_lr0.098_l21.55_leaf63,1285.853635,1124.440418,22.0,0.098421,800,1.552632,63,10.0,30


stored at: random_search_fe_fs_results_85.csv
 log in: random_search_fe_fs_results_85.log


##### __75%__

In [ ]:
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 100
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.75

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results_75.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results_75.csv", index=False)
print("stored at: random_search_fe_fs_results_75.csv")
print(" log in: random_search_fe_fs_results_75.log")



-- doing 1/100
   >> [R0_iter1000_lr0.135_l21.97_leaf31] Train MAE: 1180.45 | Val MAE: 1304.01 | Sel: 19 feats
   >> [R1_iter1000_lr0.091_l20.92_leaf63] Train MAE: 1128.71 | Val MAE: 1285.48 | Sel: 19 feats
   >> [R2_iter1500_lr0.102_l20.71_leaf63] Train MAE: 1072.85 | Val MAE: 1270.03 | Sel: 19 feats
   >> [R3_iter600_lr0.087_l21.03_leaf63] Train MAE: 1140.61 | Val MAE: 1290.58 | Sel: 19 feats
   >> [R4_iter1000_lr0.135_l22.18_leaf31] Train MAE: 1188.32 | Val MAE: 1307.31 | Sel: 19 feats

-- doing 6/100
   >> [R5_iter1500_lr0.146_l22.50_leaf127] Train MAE: 1031.09 | Val MAE: 1265.65 | Sel: 19 feats
   >> [R6_iter1000_lr0.139_l20.61_leaf31] Train MAE: 1150.44 | Val MAE: 1296.03 | Sel: 19 feats
   >> [R7_iter600_lr0.095_l21.87_leaf63] Train MAE: 1083.03 | Val MAE: 1266.26 | Sel: 19 feats
   >> [R8_iter1500_lr0.091_l20.50_leaf31] Train MAE: 1164.07 | Val MAE: 1298.15 | Sel: 19 feats
   >> [R9_iter800_lr0.132_l22.18_leaf127] Train MAE: 1015.12 | Val MAE: 1264.00 | Sel: 19 feats

-- doing

,config,val_mae_mean,train_mae_mean,avg_selected_features,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
32,R32_iter800_lr0.095_l21.76_leaf127,1255.900728,1015.664073,19.0,0.094737,800,1.763158,127,20.0,20
23,R23_iter600_lr0.091_l22.08_leaf127,1258.573143,1029.943084,19.0,0.091053,600,2.078947,127,15.0,20
30,R30_iter600_lr0.132_l20.92_leaf127,1259.392651,1009.426981,19.0,0.131579,600,0.921053,127,20.0,20
91,R91_iter600_lr0.109_l21.03_leaf127,1261.156148,1031.844597,19.0,0.109474,600,1.026316,127,15.0,20
94,R94_iter800_lr0.080_l20.61_leaf127,1262.365200,1055.133620,19.0,0.080000,800,0.605263,127,15.0,30
68,R68_iter1000_lr0.102_l22.39_leaf127,1262.616877,1045.177011,19.0,0.102105,1000,2.394737,127,20.0,30
99,R99_iter600_lr0.132_l22.29_leaf127,1263.048537,1025.556081,19.0,0.131579,600,2.289474,127,20.0,20
48,R48_iter800_lr0.091_l22.50_leaf63,1263.392331,1066.654068,19.0,0.091053,800,2.500000,63,15.0,20
44,R44_iter800_lr0.084_l21.87_leaf127,1263.521698,1048.792860,19.0,0.083684,800,1.868421,127,15.0,20
9,R9_iter800_lr0.132_l22.18_leaf127,1263.999785,1015.120707,19.0,0.131579,800,2.184211,127,25.0,30


stored at: random_search_fe_fs_results_75.csv
 log in: random_search_fe_fs_results_75.log


##### __80%__

In [ ]:
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 100
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.8

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results_80.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results_80.csv", index=False)
print("stored at: random_search_fe_fs_results_80.csv")
print(" log in: random_search_fe_fs_results_80.log")



-- doing 1/100
   >> [R0_iter800_lr0.121_l21.45_leaf127] Train MAE: 1036.81 | Val MAE: 1264.29 | Sel: 20 feats
   >> [R1_iter600_lr0.080_l22.08_leaf63] Train MAE: 1131.76 | Val MAE: 1285.66 | Sel: 20 feats
   >> [R2_iter600_lr0.091_l21.55_leaf31] Train MAE: 1196.95 | Val MAE: 1311.68 | Sel: 20 feats
   >> [R3_iter800_lr0.087_l22.29_leaf31] Train MAE: 1213.84 | Val MAE: 1320.26 | Sel: 20 feats
   >> [R4_iter1000_lr0.098_l21.03_leaf127] Train MAE: 1039.67 | Val MAE: 1259.13 | Sel: 20 feats

-- doing 6/100
   >> [R5_iter1500_lr0.098_l20.82_leaf127] Train MAE: 1097.56 | Val MAE: 1280.49 | Sel: 20 feats
   >> [R6_iter600_lr0.087_l21.03_leaf31] Train MAE: 1208.61 | Val MAE: 1316.05 | Sel: 20 feats
   >> [R7_iter1000_lr0.121_l22.50_leaf63] Train MAE: 1114.01 | Val MAE: 1282.69 | Sel: 20 feats
   >> [R8_iter800_lr0.146_l20.71_leaf31] Train MAE: 1178.16 | Val MAE: 1309.55 | Sel: 20 feats
   >> [R9_iter600_lr0.135_l21.76_leaf31] Train MAE: 1172.03 | Val MAE: 1301.57 | Sel: 20 feats

-- doing 11

,config,val_mae_mean,train_mae_mean,avg_selected_features,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
74,R74_iter1000_lr0.095_l21.13_leaf127,1250.814305,998.037088,20.0,0.094737,1000,1.131579,127,NaN,20
24,R24_iter800_lr0.098_l20.50_leaf127,1257.213990,1027.283721,20.0,0.098421,800,0.500000,127,15.0,20
83,R83_iter1000_lr0.080_l21.76_leaf127,1257.267903,1028.851515,20.0,0.080000,1000,1.763158,127,NaN,30
35,R35_iter600_lr0.080_l20.50_leaf127,1257.896845,1035.049646,20.0,0.080000,600,0.500000,127,20.0,30
15,R15_iter1500_lr0.095_l20.50_leaf127,1258.239632,1022.641400,20.0,0.094737,1500,0.500000,127,15.0,20
87,R87_iter800_lr0.109_l22.29_leaf127,1258.436234,1020.237767,20.0,0.109474,800,2.289474,127,NaN,20
55,R55_iter1000_lr0.121_l21.87_leaf127,1258.524347,1013.367800,20.0,0.120526,1000,1.868421,127,25.0,20
56,R56_iter600_lr0.117_l22.39_leaf127,1258.754556,1028.059346,20.0,0.116842,600,2.394737,127,20.0,30
40,R40_iter1000_lr0.098_l22.18_leaf127,1259.067651,1021.625841,20.0,0.098421,1000,2.184211,127,20.0,20
4,R4_iter1000_lr0.098_l21.03_leaf127,1259.133364,1039.666950,20.0,0.098421,1000,1.026316,127,NaN,30


stored at: random_search_fe_fs_results_80.csv
 log in: random_search_fe_fs_results_80.log


##### __65%__

In [ ]:
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 50
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.65

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results_65.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results_65.csv", index=False)
print("stored at: random_search_fe_fs_results_65.csv")
print(" log in: random_search_fe_fs_results_65.log")



-- doing 1/50
   >> [R0_iter1000_lr0.080_l22.18_leaf127] Train MAE: 1047.06 | Val MAE: 1261.41 | Sel: 17 feats
   >> [R1_iter600_lr0.143_l21.34_leaf127] Train MAE: 1016.30 | Val MAE: 1261.05 | Sel: 17 feats
   >> [R2_iter1000_lr0.091_l20.50_leaf127] Train MAE: 1073.02 | Val MAE: 1268.02 | Sel: 17 feats
   >> [R3_iter1000_lr0.132_l20.71_leaf127] Train MAE: 1054.86 | Val MAE: 1272.51 | Sel: 17 feats
   >> [R4_iter600_lr0.146_l20.50_leaf127] Train MAE: 1031.84 | Val MAE: 1265.95 | Sel: 17 feats

-- doing 6/50
   >> [R5_iter800_lr0.117_l20.92_leaf31] Train MAE: 1174.86 | Val MAE: 1301.93 | Sel: 17 feats
   >> [R6_iter1000_lr0.150_l22.50_leaf63] Train MAE: 1088.01 | Val MAE: 1278.55 | Sel: 17 feats
   >> [R7_iter600_lr0.098_l20.50_leaf31] Train MAE: 1216.49 | Val MAE: 1321.52 | Sel: 17 feats
   >> [R8_iter1500_lr0.128_l21.87_leaf31] Train MAE: 1159.14 | Val MAE: 1293.35 | Sel: 17 feats
   >> [R9_iter800_lr0.146_l21.87_leaf63] Train MAE: 1102.32 | Val MAE: 1281.97 | Sel: 17 feats

-- doing 

,config,val_mae_mean,train_mae_mean,avg_selected_features,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
44,R44_iter1000_lr0.113_l21.13_leaf127,1258.527405,1019.545063,17.0,0.113158,1000,1.131579,127,25.0,20
31,R31_iter600_lr0.109_l20.50_leaf127,1259.715016,1015.226144,17.0,0.109474,600,0.500000,127,15.0,20
20,R20_iter1500_lr0.121_l21.55_leaf127,1259.786309,1019.074451,17.0,0.120526,1500,1.552632,127,25.0,30
36,R36_iter1500_lr0.106_l20.92_leaf127,1260.764864,1033.791004,17.0,0.105789,1500,0.921053,127,25.0,30
1,R1_iter600_lr0.143_l21.34_leaf127,1261.049514,1016.298928,17.0,0.142632,600,1.342105,127,25.0,20
46,R46_iter800_lr0.139_l21.97_leaf127,1261.125917,1018.703741,17.0,0.138947,800,1.973684,127,20.0,20
0,R0_iter1000_lr0.080_l22.18_leaf127,1261.414278,1047.062383,17.0,0.080000,1000,2.184211,127,25.0,30
32,R32_iter800_lr0.106_l21.55_leaf127,1262.695340,1042.179948,17.0,0.105789,800,1.552632,127,20.0,30
45,R45_iter600_lr0.080_l22.39_leaf127,1263.377982,1055.538162,17.0,0.080000,600,2.394737,127,NaN,40
49,R49_iter1000_lr0.087_l20.82_leaf127,1263.534816,1048.879433,17.0,0.087368,1000,0.815789,127,NaN,40


stored at: random_search_fe_fs_results_65.csv
 log in: random_search_fe_fs_results_65.log


##### __60%__

In [ ]:
import numpy as np
import pandas as pd
import random
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) CONFIGS GERAIS
# ---------------------------------------------------------
N_SPLITS = 8
N_ITER = 15
RANDOM_STATE = 42

# Percentagem de features a manter no SelectFromModel
FS_KEEP_RATIO = 0.60

# RandomForest para Feature Selection (podes ajustar)
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 0.1) CONFIGURAÇÃO DO LOGGING
# ---------------------------------------------------------

logging.basicConfig(
    filename="random_search_fe_fs_results_60.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("- 10 iterations")

# ---------------------------------------------------------
# 0.2) CONFIGURAÇÃO DO RANDOM SEARCH
# ---------------------------------------------------------
param_distributions = {
    "min_samples_leaf": [20, 30, 40],
    "max_leaf_nodes": [31, 63, 127],
    "max_iter": [600, 800, 1000, 1500],
    "max_depth": [None, 10, 15, 20, 25],
    "learning_rate": np.linspace(0.08, 0.15, 20),
    "l2_regularization": np.linspace(0.5, 2.5, 20),
    "max_bins": [255],
    "random_state": [RANDOM_STATE],
}

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())

    for i in range(n_iter):
        params = {}
        for key in keys:
            params[key] = random.choice(list(param_distributions[key]))

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

CONFIGS = get_random_configs(N_ITER)

# ---------------------------------------------------------
# 1) K-FOLD E SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Assume que já tens no notebook:
# - X, y
# - numeric_features, categorical_features
# - fill_unknown, column_string_transformer
# - fit_* e transform_* (os teus passos)
# - valid_brands, valid_models_by_brand, valid_transmissions, valid_fueltypes
# - MyTargetEncoder, MyOneHotEncoder

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df, column=col, remove_middle_spaces=True, allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 1.1) FEATURE ENGINEERING (FUNÇÕES)
# ---------------------------------------------------------
def create_age_and_drop_year(df, year_col="year", base_year=2020, clip_future=True):
    df = df.copy()
    year = pd.to_numeric(df[year_col], errors="coerce")
    age = base_year - year
    if clip_future:
        age = age.clip(lower=0)
    df["age"] = age
    df = df.drop(columns=[year_col])
    return df

def add_owners_flagged(
    df,
    owners_col="previousOwners",
    new_col="owners_flagged",
    drop_original=True,
    na_as_zero=True
):
    df = df.copy()
    owners = pd.to_numeric(df[owners_col], errors="coerce")

    if na_as_zero:
        owners_filled = owners.fillna(0)
        df[new_col] = (owners_filled > 1).astype("int8")
    else:
        df[new_col] = (owners > 1).fillna(False).astype("int8")

    if drop_original and owners_col in df.columns:
        df = df.drop(columns=[owners_col])

    return df

def add_mileage_features(df, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True):
    df = df.copy()

    mileage = pd.to_numeric(df[mileage_col], errors="coerce")
    mileage = mileage.where(mileage >= 0, np.nan)

    df["log_mileage"] = np.log1p(mileage)

    age = pd.to_numeric(df[age_col], errors="coerce")
    age_safe = age.clip(lower=1)  # só para divisão
    df["miles_per_year"] = mileage / age_safe
    df["log_miles_per_year"] = np.log1p(df["miles_per_year"])

    if drop_ratio and "miles_per_year" in df.columns:
        df = df.drop(columns=["miles_per_year"])

    if drop_original and mileage_col in df.columns:
        df = df.drop(columns=[mileage_col])

    return df

def add_engine_bins(df, engine_col="engineSize", new_col="engine_bin", bins=None):
    df = df.copy()

    engine = pd.to_numeric(df[engine_col], errors="coerce")
    engine = engine.where(engine >= 0, np.nan)

    if bins is None:
        bins = [0, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 4.0, np.inf]

    df[new_col] = pd.cut(engine, bins=bins, include_lowest=True, labels=False).astype("Int64")
    return df

# ---------------------------------------------------------
# 1.2) EVAL DE 1 CONFIG
# ---------------------------------------------------------
def eval_one_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # TARGET (LOG)
        y_train_log = np.log1p(y_train)

        # Normalizar categóricas base
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Restringir às colunas base (mantém previousOwners e mileage para cleaning)
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        # engine_bin como low-card para OHE (robusto a NaNs)
        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        # alinhar colunas
        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # D.1) FEATURE SELECTION (SelectFromModel + RandomForest)
        #     - threshold=-np.inf
        #     - max_features = 80% das features
        # -----------------------------------------------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        support = selector.get_support()
        selected_cols = X_train_final.columns[support]

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # -----------------------------------------------------
        # E) TREINO (HGB)
        # -----------------------------------------------------
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": n_feats,
            "n_features_selected": len(selected_cols),
            "train_mae": mae_tr,
            "val_mae": mae_val
        })

    df_folds = pd.DataFrame(fold_rows)
    mean_mae_tr  = df_folds["train_mae"].mean()
    mean_mae_val = df_folds["val_mae"].mean()
    mean_sel     = df_folds["n_features_selected"].mean()

    print(f"   >> [{name}] Train MAE: {mean_mae_tr:.2f} | Val MAE: {mean_mae_val:.2f} | Sel: {mean_sel:.0f} feats")
    logging.info(
        f"{name} | Train MAE: {mean_mae_tr:.4f} | Val MAE: {mean_mae_val:.4f} | "
        f"AvgSelected: {mean_sel:.1f} | Params: {params}"
    )

    out = {
        "config": name,
        "val_mae_mean": mean_mae_val,
        "train_mae_mean": mean_mae_tr,
        "avg_selected_features": float(mean_sel),
        **params
    }
    return out

# ---------------------------------------------------------
# 2) CORRER TODAS AS CONFIGS
# ---------------------------------------------------------
results = []
for i, (name, params) in enumerate(CONFIGS):
    if i % 5 == 0:
        print(f"\n-- doing {i+1}/{len(CONFIGS)}")

    res = eval_one_config(name, params)
    results.append(res)

results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# 3) RESULTADOS FINAIS
# ---------------------------------------------------------
print("\n- 10 BEST Configurations")
results_df_sorted = results_df.sort_values("val_mae_mean", ascending=True)

cols_show = [
    "config",
    "val_mae_mean", "train_mae_mean",
    "avg_selected_features",
    "learning_rate", "max_iter", "l2_regularization", "max_leaf_nodes", "max_depth", "min_samples_leaf"
]

display(results_df_sorted[cols_show].head(10))

results_df_sorted.to_csv("random_search_fe_fs_results_60.csv", index=False)
print("stored at: random_search_fe_fs_results_60.csv")
print(" log in: random_search_fe_fs_results_60.log")



-- doing 1/15
   >> [R0_iter1500_lr0.121_l21.45_leaf63] Train MAE: 1106.51 | Val MAE: 1283.89 | Sel: 15 feats
   >> [R1_iter1500_lr0.135_l21.76_leaf63] Train MAE: 1117.38 | Val MAE: 1285.01 | Sel: 15 feats
   >> [R2_iter600_lr0.102_l22.08_leaf63] Train MAE: 1112.34 | Val MAE: 1280.45 | Sel: 15 feats
   >> [R3_iter1500_lr0.102_l21.87_leaf63] Train MAE: 1153.44 | Val MAE: 1298.55 | Sel: 15 feats
   >> [R4_iter1000_lr0.128_l20.61_leaf31] Train MAE: 1141.06 | Val MAE: 1292.35 | Sel: 15 feats

-- doing 6/15
   >> [R5_iter800_lr0.091_l22.08_leaf127] Train MAE: 1088.28 | Val MAE: 1275.84 | Sel: 15 feats
   >> [R6_iter600_lr0.143_l21.66_leaf31] Train MAE: 1194.94 | Val MAE: 1313.93 | Sel: 15 feats
   >> [R7_iter800_lr0.106_l22.18_leaf63] Train MAE: 1111.96 | Val MAE: 1285.82 | Sel: 15 feats
   >> [R8_iter800_lr0.143_l22.29_leaf31] Train MAE: 1185.84 | Val MAE: 1310.23 | Sel: 15 feats
   >> [R9_iter1500_lr0.135_l22.18_leaf31] Train MAE: 1149.84 | Val MAE: 1295.84 | Sel: 15 feats

-- doing 11/1

,config,val_mae_mean,train_mae_mean,avg_selected_features,learning_rate,max_iter,l2_regularization,max_leaf_nodes,max_depth,min_samples_leaf
5,R5_iter800_lr0.091_l22.08_leaf127,1275.840313,1088.276340,15.0,0.091053,800,2.078947,127,15.0,40
11,R11_iter1000_lr0.095_l20.82_leaf63,1278.359416,1099.218006,15.0,0.094737,1000,0.815789,63,20.0,20
2,R2_iter600_lr0.102_l22.08_leaf63,1280.450074,1112.344579,15.0,0.102105,600,2.078947,63,20.0,40
0,R0_iter1500_lr0.121_l21.45_leaf63,1283.885507,1106.507437,15.0,0.120526,1500,1.447368,63,10.0,20
1,R1_iter1500_lr0.135_l21.76_leaf63,1285.009472,1117.376320,15.0,0.135263,1500,1.763158,63,25.0,40
7,R7_iter800_lr0.106_l22.18_leaf63,1285.822965,1111.955653,15.0,0.105789,800,2.184211,63,25.0,40
14,R14_iter800_lr0.102_l20.71_leaf127,1289.153572,1121.269383,15.0,0.102105,800,0.710526,127,10.0,30
4,R4_iter1000_lr0.128_l20.61_leaf31,1292.348023,1141.062070,15.0,0.127895,1000,0.605263,31,10.0,20
9,R9_iter1500_lr0.135_l22.18_leaf31,1295.838672,1149.835597,15.0,0.135263,1500,2.184211,31,25.0,20
13,R13_iter800_lr0.150_l22.50_leaf31,1296.312575,1146.289248,15.0,0.150000,800,2.500000,31,15.0,20


stored at: random_search_fe_fs_results_60.csv
 log in: random_search_fe_fs_results_60.log


## 5. Final Configuration <a id="finalconfig"></a>

### 5.1. CV

In [ ]:
import os
import numpy as np
import pandas as pd
import logging

from IPython.display import display
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# 0) FIXED CONFIG (YOUR BEST PARAMS)
# ---------------------------------------------------------
final_params = {
    "min_samples_leaf": 20,
    "max_leaf_nodes": 127,
    "max_iter": 600,
    "max_depth": 25,
    "learning_rate": float(np.float64(0.09105263157894737)),
    "l2_regularization": float(np.float64(1.4473684210526314)),
    "max_bins": 255,
    "random_state": 42,
}

# ---------------------------------------------------------
# 1) GLOBAL SETTINGS
# ---------------------------------------------------------
N_SPLITS = 8
RANDOM_STATE = 42

FS_KEEP_RATIO = 0.65  # keep top 90% features by RF importance
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 2) OUTPUTS + LOGGING (JUPYTER-SAFE)
# ---------------------------------------------------------
OUT_DIR = os.path.abspath("./outputs")
os.makedirs(OUT_DIR, exist_ok=True)

LOG_PATH = os.path.join(OUT_DIR, "config_cv.log")
logging.basicConfig(
    filename=LOG_PATH,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info("cv started")
print(f"- output dir: {OUT_DIR}")
print(f"- log file: {LOG_PATH}")

# ---------------------------------------------------------
# 3) CV SETUP
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)



cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(df, column=col, remove_middle_spaces=True, allow_extra_chars="")
    return df


# ---------------------------------------------------------
# 5) CV EVALUATION FOR ONE FIXED CONFIG
# ---------------------------------------------------------
def run_cv_for_params(name: str, params: dict):
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        y_train_log = np.log1p(y_train)

        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # A) cleaning (fit on train, apply on val)
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        # B) resolvers
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # C) feature engineering
        X_train = add_owners_flagged(X_train, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   owners_col="previousOwners", new_col="owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = add_mileage_features(X_train, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, engine_col="engineSize", new_col="engine_bin")
        X_val   = add_engine_bins(X_val,   engine_col="engineSize", new_col="engine_bin")

        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # D) encoding
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_features_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_features_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features_curr],   X_val_cat],   axis=1)

        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # D.1) feature selection (RF -> SelectFromModel)
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        selected_cols = X_train_final.columns[selector.get_support()]
        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        # E) train HGB
        model = HistGradientBoostingRegressor(**params)
        model.fit(X_train_sel, y_train_log)

        pred_tr_log  = model.predict(X_train_sel)
        pred_val_log = model.predict(X_val_sel)

        pred_tr  = np.expm1(pred_tr_log)
        pred_val = np.expm1(pred_val_log)

        mae_tr  = mean_absolute_error(y_train, pred_tr)
        mae_val = mean_absolute_error(y_val,   pred_val)

        fold_rows.append({
            "config": name,
            "fold": fold,
            "n_features_total": int(n_feats),
            "n_features_selected": int(len(selected_cols)),
            "train_mae": float(mae_tr),
            "val_mae": float(mae_val),
        })

        print(f"- fold {fold} | train_mae={mae_tr:.2f} | val_mae={mae_val:.2f} | selected={len(selected_cols)}")
        logging.info(f"fold {fold} | train_mae={mae_tr:.6f} | val_mae={mae_val:.6f} | selected={len(selected_cols)}")

    folds_df = pd.DataFrame(fold_rows)

    summary = {
        "config": name,
        "train_mae_mean": float(folds_df["train_mae"].mean()),
        "train_mae_std":  float(folds_df["train_mae"].std(ddof=1)),
        "val_mae_mean":   float(folds_df["val_mae"].mean()),
        "val_mae_std":    float(folds_df["val_mae"].std(ddof=1)),
        "selected_mean":  float(folds_df["n_features_selected"].mean()),
        "selected_std":   float(folds_df["n_features_selected"].std(ddof=1)),
        **params
    }

    print(f"- done | train_mae_mean={summary['train_mae_mean']:.2f} | val_mae_mean={summary['val_mae_mean']:.2f}")
    logging.info(f"done | train_mae_mean={summary['train_mae_mean']:.6f} | val_mae_mean={summary['val_mae_mean']:.6f}")

    return folds_df, pd.DataFrame([summary])

# ---------------------------------------------------------
# 6) RUN
# ---------------------------------------------------------
print("- starting CV for final_params")
logging.info(f"config params: {final_params}")

folds_df, summary_df = run_cv_for_params("final_config", final_params)

display(folds_df)
display(summary_df)

FOLDS_CSV = os.path.join(OUT_DIR, "final_config_cv_folds.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "final_config_cv_summary.csv")

folds_df.to_csv(FOLDS_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)

print(f"- saved folds: {FOLDS_CSV}")
print(f"- saved summary: {SUMMARY_CSV}")
print(f"- log file: {LOG_PATH}")

logging.info(f"saved folds: {FOLDS_CSV}")
logging.info(f"saved summary: {SUMMARY_CSV}")
logging.info("cv finished")
logging.shutdown()


- output dir: /Users/franciscafernandes/Desktop/MachineLearning/projeto/Machine-Learning-G51/notebooks/model_assessment_optimazation/outputs
- log file: /Users/franciscafernandes/Desktop/MachineLearning/projeto/Machine-Learning-G51/notebooks/model_assessment_optimazation/outputs/config_cv.log
- starting CV for final_params
- fold 1 | train_mae=1073.43 | val_mae=1260.50 | selected=17
- fold 2 | train_mae=1072.35 | val_mae=1285.75 | selected=17
- fold 3 | train_mae=1064.59 | val_mae=1283.89 | selected=17
- fold 4 | train_mae=1024.11 | val_mae=1290.84 | selected=17
- fold 5 | train_mae=983.87 | val_mae=1249.15 | selected=17
- fold 6 | train_mae=987.28 | val_mae=1247.20 | selected=17
- fold 7 | train_mae=1020.06 | val_mae=1235.25 | selected=17
- fold 8 | train_mae=1034.28 | val_mae=1228.62 | selected=17
- done | train_mae_mean=1032.50 | val_mae_mean=1260.15


,config,fold,n_features_total,n_features_selected,train_mae,val_mae
0,final_config,1,25,17,1073.433847,1260.504796
1,final_config,2,25,17,1072.354716,1285.747898
2,final_config,3,25,17,1064.585480,1283.886655
3,final_config,4,25,17,1024.112416,1290.843950
4,final_config,5,25,17,983.873171,1249.154135
5,final_config,6,25,17,987.278480,1247.196078
6,final_config,7,25,17,1020.059804,1235.252404
7,final_config,8,25,17,1034.282904,1228.620799


,config,train_mae_mean,train_mae_std,val_mae_mean,val_mae_std,selected_mean,selected_std,min_samples_leaf,max_leaf_nodes,max_iter,max_depth,learning_rate,l2_regularization,max_bins,random_state
0,final_config,1032.497602,35.719741,1260.15084,24.093922,17.0,0.0,20,127,600,25,0.091053,1.447368,255,42


- saved folds: /Users/franciscafernandes/Desktop/MachineLearning/projeto/Machine-Learning-G51/notebooks/model_assessment_optimazation/outputs/final_config_cv_folds.csv
- saved summary: /Users/franciscafernandes/Desktop/MachineLearning/projeto/Machine-Learning-G51/notebooks/model_assessment_optimazation/outputs/final_config_cv_summary.csv
- log file: /Users/franciscafernandes/Desktop/MachineLearning/projeto/Machine-Learning-G51/notebooks/model_assessment_optimazation/outputs/config_cv.log


### 5.2. Final Model and Kaggle Submission

In [ ]:

# ==============================================================================
# 0) SETTINGS
# ==============================================================================
RANDOM_STATE = 42

final_params = {
    "min_samples_leaf": 20,
    "max_leaf_nodes": 127,
    "max_iter": 600,
    "max_depth": 25,
    "learning_rate": float(np.float64(0.09105263157894737)),
    "l2_regularization": float(np.float64(1.4473684210526314)),
    "max_bins": 255,
    "random_state": 42,
}

FS_KEEP_RATIO = 0.65
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

print(f"- using params: {final_params}")
print(f"- FS_KEEP_RATIO: {FS_KEEP_RATIO}")

# ==============================================================================
# 1) LOAD TEST + IDS
# ==============================================================================
try:
    test_df_raw = pd.read_csv("test.csv")
except:
    test_df_raw = pd.read_csv("../../project_data/test.csv")

ID_CANDIDATES = ["id", "carID", "carId", "car_id"]
ID_COL = next((c for c in ID_CANDIDATES if c in test_df_raw.columns), None)

if ID_COL is not None:
    test_ids = test_df_raw[ID_COL].copy()
    test_df = test_df_raw.drop(columns=[ID_COL]).copy()
else:
    test_ids = test_df_raw.index.copy()
    test_df = test_df_raw.copy()

# ==============================================================================
# 2) START DATA
# ==============================================================================
X_full = X.copy()
y_full = y.copy()
y_full_log = np.log1p(y_full)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]

# ==============================================================================
# 4) STRING NORMALIZATION (TRAIN + TEST)
# ==============================================================================
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer(X_full, column=col, remove_middle_spaces=True, allow_extra_chars="")
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(test_df, column=col, remove_middle_spaces=True, allow_extra_chars="")

# Restrict to expected columns (train schema)
expected_cols = [c for c in (numeric_features + categorical_features) if c in X_full.columns]
X_full = X_full[expected_cols].copy()
X_test = test_df.reindex(columns=expected_cols, fill_value=np.nan).copy()

# ==============================================================================
# 5) FIT + TRANSFORM ON FULL TRAIN
# ==============================================================================
print("- fitting transforms on full train")

year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# IMPORTANT: owners imputer must happen BEFORE dropping previousOwners
owners_state = fit_previous_owners_imputer(X_full, "previousOwners", "year", "mileage")
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 6) FEATURE ENGINEERING (FULL TRAIN)
# ==============================================================================
print("- creating engineered features on full train")

X_full = add_owners_flagged(X_full, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_full = create_age_and_drop_year(X_full, year_col="year", base_year=2020)
X_full = add_mileage_features(X_full, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_full = add_engine_bins(X_full, engine_col="engineSize", new_col="engine_bin")

# engine_bin as low-card categorical
X_full["engine_bin"] = X_full["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
low_card_curr = low_card_features + ["engine_bin"]

# ==============================================================================
# 7) ENCODING (FIT ON FULL TRAIN)
# ==============================================================================
print("- fitting encoders on full train")

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full_log)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_curr])
X_full_low = ohe.transform(X_full[low_card_curr])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr = [c for c in X_full.columns if c not in drop_for_numeric]

X_full_final = pd.concat([X_full[numeric_features_curr], X_full_cat], axis=1)
print(f"- train matrix shape: {X_full_final.shape}")

# ==============================================================================
# 8) FEATURE SELECTION (FIT ON FULL TRAIN)
# ==============================================================================
print("- fitting feature selector (RF)")

n_feats = X_full_final.shape[1]
k = int(np.ceil(FS_KEEP_RATIO * n_feats))
k = max(1, min(k, n_feats))

rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
rf_fs.fit(X_full_final, y_full_log)

selector = SelectFromModel(
    estimator=rf_fs,
    threshold=-np.inf,
    max_features=k,
    prefit=True
)

selected_cols = X_full_final.columns[selector.get_support()]
X_full_sel = X_full_final[selected_cols]
print(f"- selected features: {len(selected_cols)}/{n_feats}")

# ==============================================================================
# 9) TRAIN FINAL HGB (LOG TARGET)
# ==============================================================================
print("- training final HGB")
hgb_final = HistGradientBoostingRegressor(**final_params)
hgb_final.fit(X_full_sel, y_full_log)
print("- model trained")

# ==============================================================================
# 10) TRANSFORM TEST (TRANSFORM-ONLY) + SAME FE + SAME ENCODING + SAME FS
# ==============================================================================
print("- transforming test set")

X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_custom_rules(X_test, "tax", "year", "fuelType", "engineSize")
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test = transform_previous_owners_imputer(X_test, state=owners_state)

X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# same FE
X_test = add_owners_flagged(X_test, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_test = create_age_and_drop_year(X_test, year_col="year", base_year=2020)
X_test = add_mileage_features(X_test, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_test = add_engine_bins(X_test, engine_col="engineSize", new_col="engine_bin")

X_test["engine_bin"] = X_test["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

# encoding transform-only
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_curr])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr_test = [c for c in X_test.columns if c not in drop_for_numeric]

X_test_final = pd.concat([X_test[numeric_features_curr_test], X_test_cat], axis=1)

# align to train columns before selecting
X_test_final = X_test_final.reindex(columns=X_full_final.columns, fill_value=0)

# apply same selected columns
X_test_sel = X_test_final.reindex(columns=selected_cols, fill_value=0)

print(f"- test matrix shape: {X_test_sel.shape}")

# ==============================================================================
# 11) PREDICT + SUBMISSION
# ==============================================================================
pred_log = hgb_final.predict(X_test_sel)
pred_final = np.expm1(pred_log)
pred_final = np.maximum(pred_final, 0)

submission = pd.DataFrame({
    (ID_COL if ID_COL is not None else "id"): test_ids,
    "price": pred_final
})

submission_filename = "submission_hgb_log_target_fs5.csv"
submission.to_csv(submission_filename, index=False)

print(f"- saved: {submission_filename}")
print(submission.head())


- using params: {'min_samples_leaf': 20, 'max_leaf_nodes': 127, 'max_iter': 600, 'max_depth': 25, 'learning_rate': 0.09105263157894737, 'l2_regularization': 1.4473684210526314, 'max_bins': 255, 'random_state': 42}
- FS_KEEP_RATIO: 0.65
- fitting transforms on full train
- creating engineered features on full train
- fitting encoders on full train
- train matrix shape: (75973, 25)
- fitting feature selector (RF)
- selected features: 17/25
- training final HGB
- model trained
- transforming test set
- test matrix shape: (32567, 17)
- saved: submission_hgb_log_target_fs5.csv
    carID         price
0   89856   6490.349255
1  106581  23638.774903
2   80886  14125.008808
3  100174  16889.942184
4   81376  22648.157405


### 5.3. Visualization and Final Regards about the Implementation